In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision.transforms import v2
import timm
from omegaconf import DictConfig
from datasets import load_dataset
import lightning as L
from torchvision.ops import MLP
# Import for TensorBoard
from lightning.pytorch.loggers import TensorBoardLogger
from torch.optim.lr_scheduler import LinearLR, CosineAnnealingLR, SequentialLR

In [3]:
from modules.loss.univariate import EppsPulley
from modules.loss.multivariate import SlicingUnivariateTest
univariate_test = EppsPulley(n_points=17)
# Create the multivariate slicing test
loss_fn = SlicingUnivariateTest(
    univariate_test=univariate_test, 
    num_slices=1024
)

In [4]:
# --- Original Classes ---

class SIGReg(torch.nn.Module):
    def __init__(self, knots=17):
        super().__init__()
        t = torch.linspace(0, 3, knots, dtype=torch.float32)
        dt = 3 / (knots - 1)
        weights = torch.full((knots,), 2 * dt, dtype=torch.float32)
        weights[[0, -1]] = dt
        window = torch.exp(-t.square() / 2.0)
        self.register_buffer("t", t)
        self.register_buffer("phi", window)
        self.register_buffer("weights", weights * window)

    def forward(self, proj):
        # A will be automatically moved to the correct device by Lightning
        A = torch.randn(proj.size(-1), 256, device=proj.device) 
        A = A.div_(A.norm(p=2, dim=0))
        x_t = (proj @ A).unsqueeze(-1) * self.t
        err = (x_t.cos().mean(-3) - self.phi).square() + x_t.sin().mean(-3).square()
        statistic = (err @ self.weights) * proj.size(-2)
        return statistic.mean()


class ViTEncoder(nn.Module):
    def __init__(self, proj_dim=128):
        super().__init__()
        self.backbone = timm.create_model(
            "vit_small_patch8_224",
            pretrained=False,
            num_classes=512,
            drop_path_rate=0.1,
            img_size=128,
        )
        self.proj = MLP(512, [2048, 2048, proj_dim], norm_layer=nn.BatchNorm1d) # Jepa Projector

    def forward(self, x):
        N, V = x.shape[:2]
        emb = self.backbone(x.flatten(0, 1))
        # N, V, -1 -> V, N, -1
        return emb, self.proj(emb).reshape(N, V, -1).transpose(0, 1) 


# --- Lightning Data Module ---

class ImageNetDataModule(L.LightningDataModule):
    def __init__(self, cfg: DictConfig):
        super().__init__()
        self.cfg = cfg
        self.save_hyperparameters()

        self.aug = v2.Compose(
            [
                v2.RandomResizedCrop(128, scale=(0.08, 1.0)),
                v2.RandomApply([v2.ColorJitter(0.8, 0.8, 0.8, 0.2)], p=0.8),
                v2.RandomGrayscale(p=0.2),
                v2.RandomApply([v2.GaussianBlur(kernel_size=7, sigma=(0.1, 2.0))]),
                v2.RandomApply([v2.RandomSolarize(threshold=128)], p=0.2),
                v2.RandomHorizontalFlip(),
                v2.ToImage(),
                v2.ToDtype(torch.float32, scale=True),
                v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
            ]
        )
        self.test_transform = v2.Compose(
            [
                v2.Resize(128),
                v2.CenterCrop(128),
                v2.ToImage(),
                v2.ToDtype(torch.float32, scale=True),
                v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
            ]
        )

    class HFDataset(torch.utils.data.Dataset):
        def __init__(self, ds, transform, V):
            self.ds = ds
            self.transform = transform
            self.V = V

        def __getitem__(self, i):
            item = self.ds[i]
            img = item["image"].convert("RGB")
            # Create V views using the provided transform
            return torch.stack([self.transform(img) for _ in range(self.V)]), item["label"]

        def __len__(self):
            return len(self.ds)

    def prepare_data(self):
        # Download dataset
        load_dataset("frgfm/imagenette", "160px", split="train")
        load_dataset("frgfm/imagenette", "160px", split="validation")

    def setup(self, stage=None):
        ds_train = load_dataset("frgfm/imagenette", "160px", split="train")
        ds_val = load_dataset("frgfm/imagenette", "160px", split="validation")

        self.train_ds = self.HFDataset(ds_train, self.aug, V=self.cfg.V)
        self.val_ds = self.HFDataset(ds_val, self.test_transform, V=1)

    def train_dataloader(self):
        return DataLoader(
            self.train_ds,
            batch_size=self.cfg.bs,
            shuffle=True,
            drop_last=True,
            num_workers=self.cfg.num_workers,
        )

    def val_dataloader(self):
        # Val batch size set to 256 as in original code
        return DataLoader(
            self.val_ds,
            batch_size=256,
            num_workers=self.cfg.num_workers,
        )

In [5]:
from omegaconf import DictConfig
cfg = DictConfig({'V':4, 'bs':64, 'num_workers':8, 'epochs':800,'proj_dim':16,'lr':2e-3,'lamb':0.02,'accum_steps':4})
dm=ImageNetDataModule(cfg)
dm.setup()



In [7]:
ds=load_dataset("frgfm/imagenette", "160px", split="train")


In [9]:
ds.features

{'image': Image(mode=None, decode=True, id=None),
 'label': ClassLabel(names=['tench', 'English springer', 'cassette player', 'chain saw', 'church', 'French horn', 'garbage truck', 'gas pump', 'golf ball', 'parachute'], id=None)}

In [6]:
for batch in dm.train_dataloader():
    print(batch[0].shape, batch[1].shape)
    break

torch.Size([64, 4, 3, 128, 128]) torch.Size([64])


In [ ]:

# --- Lightning Module ---
class LeJEPALightning(L.LightningModule):
    def __init__(self, cfg: DictConfig):
        super().__init__()
        self.cfg = cfg
        self.save_hyperparameters(cfg) # Save hyperparameters for logging/checkpointing

        # Modules and loss
        self.net = ViTEncoder(proj_dim=cfg.proj_dim)
        # Probe from original code: nn.Sequential(nn.LayerNorm(512), nn.Linear(512, 100))
        self.probe = nn.Sequential(nn.LayerNorm(512), nn.Linear(512, 100))
        self.sigreg = SIGReg()

    def forward(self, x):
        return self.net(x)

    def configure_optimizers(self):
        # Optimizer and scheduler setup from the original code
        g1 = {"params": self.net.parameters(), "lr": self.cfg.lr, "weight_decay": 5e-2}
        g2 = {"params": self.probe.parameters(), "lr": 1e-3, "weight_decay": 1e-7}
        # This will be called during training setup
        opt = torch.optim.AdamW([g1, g2])
        train_batches = len(self.trainer.datamodule.train_dataloader())
        train_steps_per_epoch = (train_batches + self.cfg.accum_steps - 1) // self.cfg.accum_steps
        
        warmup_steps = train_steps_per_epoch
        total_steps = train_steps_per_epoch * self.cfg.epochs
        
        s1 = LinearLR(opt, start_factor=0.01, total_iters=warmup_steps)
        t_max = max(1, total_steps - warmup_steps)
        s2 = CosineAnnealingLR(opt, T_max=t_max, eta_min=1e-3)
        scheduler = SequentialLR(opt, schedulers=[s1, s2], milestones=[warmup_steps])
        
        # Return optimizer and scheduler configuration
        return {
            "optimizer": opt,
            "lr_scheduler": {
                "scheduler": scheduler,
                "interval": "step",
                "frequency": 1,
            },
        }
        
    def training_step(self, batch, batch_idx):
        vs, y = batch
        emb, proj = self(vs)

        # LeJEPA loss calculation
        inv_loss = (proj.mean(0) - proj).square().mean() # makes different views similar
        sigreg_loss = self.sigreg(proj) # to regularize to gaussian
        lejepa_loss = sigreg_loss * self.cfg.lamb + inv_loss * (1 - self.cfg.lamb)
        
        # Probe loss calculation
        y_rep, yhat = y.repeat_interleave(self.cfg.V), self.probe(emb.detach()) # detach is used to only update probe
        probe_loss = F.cross_entropy(yhat, y_rep)
        
        loss = lejepa_loss + probe_loss # lejepa loss is for representations and probe for the classifier

        # Logging to TensorBoard
        # Use self.log for automatic logging, averaging, and step tracking
        self.log_dict(
            {
                "train/probe": probe_loss,
                "train/lejepa": lejepa_loss,
                "train/sigreg": sigreg_loss,
                "train/inv": inv_loss,
                "train/loss": loss,
                "lr": self.optimizers().param_groups[0]['lr'], # Log learning rate
            },
            on_step=True,
            on_epoch=True,
            prog_bar=True,
            sync_dist=True,
        )
        
        # The loss averaging and backward pass with accumulation and scaler step 
        # is handled automatically by the Lightning Trainer.

        return loss

    def validation_step(self, batch, batch_idx):
        vs, y = batch
        
        # Evaluation is done on V=1 view (as per the original test_ds setup)
        # Note: The original code used autocast inside the inference_mode block, 
        # which is automatically handled by the Trainer's precision setting.

        # The net's forward method is called implicitly by self(vs)
        emb, _ = self(vs) 
        
        yhat = self.probe(emb)
        
        val_loss = F.cross_entropy(yhat, y)
        acc = (yhat.argmax(1) == y).float().mean()

        self.log_dict(
            {
                "test/probe_loss": val_loss,
                "test/acc": acc, # Renamed to val_loss/acc for clarity in standard Lightning, but keeping 'test/acc' for consistency with original log keys
            },
            on_step=False,
            on_epoch=True,
            sync_dist=True,
        )

    def on_train_epoch_end(self, outputs, batch, batch_idx):
        torch.cuda.empty_cache()
        return super().on_train_batch_end(outputs, batch, batch_idx)
    def on_val_batch_end(self, outputs, batch, batch_idx):
        torch.cuda.empty_cache()
        return super().on_train_batch_end(outputs, batch, batch_idx)


In [ ]:
# Set seed
L.seed_everything(0)
from omegaconf import DictConfig
cfg = DictConfig({'V':4, 'bs':64, 'num_workers':8, 'epochs':800,'proj_dim':16,'lr':2e-3,'lamb':0.02,'accum_steps':4})
# 1. Setup DataModule
dm = ImageNetDataModule(cfg)
# 2. Setup LightningModule
model = LeJEPALightning(cfg)

# 3. Setup Logger (TensorBoardLogger replaces wandb.init)
logger = TensorBoardLogger("tb_logs", name="LeJEPA")
torch.set_float32_matmul_precision('high')
# 4. Setup Trainer
# Trainer handles the loops, autocast, GradScaler, and gradient accumulation
# precision="bf16-mixed" enables autocast(dtype=torch.bfloat16) and GradScaler
# accelerator="auto" and devices="auto" are best practice
# accumulate_grad_batches=model.accum_steps implements the gradient accumulation logic
trainer = L.Trainer(
    max_epochs=cfg.epochs,
    accelerator="cuda", # Force cuda as in the original code
    devices=1,
    precision="bf16-mixed", # Handles autocast and GradScaler
    logger=logger,
    accumulate_grad_batches=model.cfg.accum_steps,
)

# 5. Train the model
trainer.fit(model, dm)

Seed set to 0
Using bfloat16 Automatic Mixed Precision (AMP)
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/home/crispy/lejepa/.venv/lib/python3.12/site-packages/lightning/pytorch/utilities/model_summary/model_summary.py:242: Precision bf16-mixed is not supported by the model summary.  Estimated model size in MB will not be accurate. Using 32 bits instead.

  | Name   | Type       | Params | Mode  | FLOPs
------------------------------------------------------
0 | net    | ViTEncoder | 27.0 M | train | 0    
1 | probe  | Sequential | 52.3 K | train | 0    
2 | sigreg | SIGReg     | 0      | train | 0    
------------------------------------------------------
27.0 M    Trainable params
0         Non-trainable pa

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]